<a href="https://colab.research.google.com/github/robsterv8/Entregavel-BigData-Processing/blob/main/Aula02_Pratica_Modelagem_Olist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Prep & Transformation | Aula 2 · Prática
## Lendo e reconstruindo o modelo transacional da Olist

**Luciano Gillieron Gavinho, PhD.** · MBA em Engenharia de Dados · Universidade Presbiteriana Mackenzie

Na Aula 1 você diagnosticou os arquivos brutos da Olist. Muitos dos problemas encontrados não são acidentes: são consequências de **decisões de modelagem** do sistema transacional, ou da falta delas. Hoje você vai fazer **engenharia reversa** do modelo (descobrir, a partir dos dados, como ele foi desenhado), testar se ele respeita as **formas normais**, impor **restrições** que os arquivos CSV não têm e, por fim, remodelar o pedido como **documento JSON**, no estilo de um banco NoSQL.

**Como trabalhar**
- Os mesmos grupos da Aula 1. Cada pessoa roda o próprio notebook, e o grupo consolida as respostas.
- Comece pela **seção A · Aquecimento**: exemplos resolvidos com a sintaxe nova de hoje.
- O produto da prática é o **Modelo do Grupo**, no fim do notebook.

### Termos usados neste notebook

| Termo | O que significa |
|---|---|
| **Entidade** | Coisa do negócio sobre a qual se guardam dados: cliente, pedido, produto. Vira uma tabela. |
| **Atributo** | Característica de uma entidade: cidade do cliente, preço do item. Vira uma coluna. |
| **Relacionamento** | Ligação entre entidades: um pedido *tem* itens. |
| **Cardinalidade** | Quantas ocorrências de uma entidade se ligam a uma ocorrência da outra: **1:1** (um para um), **1:N** (um para muitos), **N:N** (muitos para muitos). |
| **PK** | *Primary Key*, chave primária: coluna ou conjunto de colunas que identifica cada linha, sem repetição e sem nulo. |
| **FK** | *Foreign Key*, chave estrangeira: coluna que aponta para a PK de outra tabela. |
| **Chave composta** | PK formada por mais de uma coluna, como (`order_id`, `order_item_id`). |
| **Restrição** (*constraint*) | Regra que o banco de dados impõe e verifica sozinho: PK, FK, `NOT NULL` (não pode ser nulo), `UNIQUE` (não pode repetir). |
| **DDL** | *Data Definition Language*: comandos SQL que criam estruturas, como `CREATE TABLE`. |
| **Dependência funcional** (X → Y) | Para cada valor de X existe um único valor de Y. Ex.: `order_id → order_status`. |
| **Normalização** | Organizar as tabelas para que cada fato fique guardado num só lugar, evitando redundância e anomalias. |
| **Formas normais** (1FN, 2FN, 3FN) | Níveis de normalização. Cada uma proíbe um tipo de redundância (explicadas no material). |
| **Anomalia de atualização** | Inconsistência que surge quando o mesmo fato está guardado em vários lugares e só alguns são alterados. |
| **NoSQL** | Família de bancos não relacionais: documento, chave-valor, colunar e grafo. |
| **JSON** | *JavaScript Object Notation*: formato de texto que guarda dados aninhados, com a estrutura junto do dado. |
| **Documento** | Registro autocontido em JSON, que pode **embutir** dados relacionados (os itens dentro do pedido). |
| **STRUCT / LIST** | Tipos do DuckDB: `STRUCT` é um registro com campos nomeados; `LIST` é uma lista de valores. Equivalem a objeto e lista em JSON. |

## 0 · Preparação do ambiente
Execute as duas células abaixo. A primeira baixa os dados do repositório da Olist com `git clone` (cópia do repositório) e instala o DuckDB com `pip` (instalador de pacotes Python). A segunda registra cada arquivo CSV como uma *view* (consulta salva com nome) e cria a função `sql()`, que executa uma consulta e devolve um DataFrame (tabela do pandas).

In [1]:
import os, subprocess
if not os.path.exists("work-at-olist-data"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/olist/work-at-olist-data.git"], check=True)
subprocess.run(["pip", "install", "-q", "duckdb"], check=False)
print(sorted(os.listdir("work-at-olist-data/datasets")))

['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


In [2]:
import duckdb, pandas as pd, json
DATA = "work-at-olist-data/datasets"
con = duckdb.connect()

FONTES = {
    "customers":  "olist_customers_dataset.csv",
    "geolocation":"olist_geolocation_dataset.csv",
    "orders":     "olist_orders_dataset.csv",
    "items":      "olist_order_items_dataset.csv",
    "payments":   "olist_order_payments_dataset.csv",
    "reviews":    "olist_order_reviews_dataset.csv",
    "products":   "olist_products_dataset.csv",
    "sellers":    "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}
for nome, arq in FONTES.items():
    con.execute(f"CREATE OR REPLACE VIEW {nome} AS SELECT * FROM read_csv_auto('{DATA}/{arq}', header=true)")

def sql(q):
    """Executa uma consulta no DuckDB e devolve um DataFrame."""
    return con.sql(q).df()

def tentar(comando):
    """Executa um comando SQL e, se o banco recusar, mostra o erro em vez de interromper o notebook."""
    try:
        con.execute(comando)
        print("OK: comando executado.")
    except Exception as e:
        print("RECUSADO PELO BANCO:", type(e).__name__)
        print(str(e)[:300])

sql("SELECT COUNT(*) AS pedidos FROM orders")

,pedidos
0,99441


## A · Aquecimento: exemplos resolvidos
Cinco exemplos prontos com a sintaxe nova de hoje. Execute e leia o resultado. Eles usam tabelas pequenas ou a tabela `sellers` (vendedores) e **não** respondem aos exercícios.

### A.1 · Medir uma cardinalidade
Quantos pagamentos tem cada pedido? Primeiro contamos por pedido; depois resumimos com mínimo, máximo e média. O máximo diz se o relacionamento é 1:1 ou 1:N.

In [12]:
sql("""
WITH por_pedido AS (
    SELECT order_id, COUNT(*) AS pagamentos
    FROM payments
    GROUP BY order_id
)
SELECT MIN(pagamentos) AS minimo, MAX(pagamentos) AS maximo, ROUND(AVG(pagamentos), 2) AS media
FROM por_pedido
""")

,minimo,maximo,media
0,1,29,1.04


### A.2 · Testar uma dependência funcional
A dependência X → Y vale quando nenhum valor de X aparece com mais de um valor de Y. O teste é sempre o mesmo: agrupar por X e contar quantos grupos têm mais de um Y distinto. Resultado 0 significa que a dependência vale **nos dados atuais** (o que não prova que valha como regra do negócio).

Aqui testamos `seller_id → seller_state`: cada vendedor tem um único estado?

In [11]:
sql("""
SELECT COUNT(*) AS violacoes
FROM (
    SELECT seller_id
    FROM sellers
    GROUP BY seller_id
    HAVING COUNT(DISTINCT seller_state) > 1
)
""")

,violacoes
0,0


### A.3 · Criar uma tabela com restrições
`CREATE TABLE` define a estrutura e as regras. Abaixo, uma tabela de exemplo com PK e `NOT NULL`. Depois tentamos inserir uma linha que viola a PK: o banco recusa. A função `tentar()` (criada na seção 0) mostra o erro sem interromper o notebook.

In [10]:
con.execute("CREATE OR REPLACE TABLE exemplo_uf (uf VARCHAR PRIMARY KEY, nome VARCHAR NOT NULL)")
tentar("INSERT INTO exemplo_uf VALUES ('SP', 'São Paulo'), ('RJ', 'Rio de Janeiro')")
tentar("INSERT INTO exemplo_uf VALUES ('SP', 'Outro nome para SP')")   # PK repetida
tentar("INSERT INTO exemplo_uf VALUES ('MG', NULL)")                   # nome nulo
sql("SELECT * FROM exemplo_uf")

OK: comando executado.
RECUSADO PELO BANCO: ConstraintException
Constraint Error: Duplicate key "uf: SP" violates primary key constraint.
RECUSADO PELO BANCO: ConstraintException
Constraint Error: NOT NULL constraint failed: exemplo_uf.nome


,uf,nome
0,SP,São Paulo
1,RJ,Rio de Janeiro


### A.4 · Uma FK em ação
A tabela `exemplo_cidade` aponta para `exemplo_uf` pela FK `uf`. Inserir uma cidade com UF existente funciona; com UF inexistente, o banco recusa. É a **integridade referencial** sendo garantida pelo próprio banco.

In [17]:
con.execute("""
CREATE OR REPLACE TABLE exemplo_cidade (
    cidade VARCHAR PRIMARY KEY,
    uf     VARCHAR NOT NULL REFERENCES exemplo_uf(uf)
)""")
tentar("INSERT INTO exemplo_cidade VALUES ('Campinas', 'SP')")
tentar("INSERT INTO exemplo_cidade VALUES ('Manaus', 'AM')")   # AM não existe em exemplo_uf
sql("SELECT * FROM exemplo_cidade")

OK: comando executado.
RECUSADO PELO BANCO: ConstraintException
Constraint Error: Violates foreign key constraint because key "uf: AM" does not exist in the referenced table


,cidade,uf
0,Campinas,SP


### A.5 · De linhas para documento JSON, e de volta
`struct_pack(campo := valor, ...)` monta um registro com campos nomeados; `list(...)` junta várias linhas numa lista; `to_json(...)` converte para texto JSON. Para ler de volta, `json_extract_string(doc, '$.campo')` extrai um valor e `json_array_length(doc, '$.lista')` conta os elementos de uma lista.

Abaixo, um documento por estado com a lista de cidades dos vendedores daquele estado (limitado a 3 cidades, para caber na tela).

In [8]:
docs = sql("""
SELECT seller_state,
       to_json(struct_pack(
           uf      := seller_state,
           cidades := list(DISTINCT seller_city ORDER BY seller_city)[1:3]
       )) AS doc
FROM sellers
WHERE seller_state IN ('AM', 'MA')
GROUP BY seller_state
""")
display(docs)
print(json.dumps(json.loads(docs.loc[0, "doc"]), indent=2, ensure_ascii=False))

con.register("docs_exemplo", docs)   # registra o DataFrame como tabela para consultar em SQL
sql("""
SELECT json_extract_string(doc, '$.uf') AS uf,
       json_array_length(doc, '$.cidades') AS qtd_cidades
FROM docs_exemplo
""")

,seller_state,doc
0,AM,"{""uf"":""AM"",""cidades"":[""manaus""]}"
1,MA,"{""uf"":""MA"",""cidades"":[""sao luis""]}"


{
  "uf": "AM",
  "cidades": [
    "manaus"
  ]
}


,uf,qtd_cidades
0,AM,1
1,MA,1


> **Resumo das ferramentas:** cardinalidade com `GROUP BY` + `MIN/MAX`; dependência funcional com `HAVING COUNT(DISTINCT y) > 1`; restrições com `CREATE TABLE ... PRIMARY KEY / REFERENCES / NOT NULL`; documentos com `struct_pack`, `list` e `to_json`. As dicas dos exercícios explicam o que mais for preciso.

## 1 · Engenharia reversa: as cardinalidades reais
Para cada relacionamento abaixo, meça **quantas linhas do lado N existem para cada linha do lado 1** (mínimo e máximo). Com isso, classifique o relacionamento como 1:1, 1:N ou N:N.

| # | Relacionamento | Pergunta |
|---|---|---|
| a | `customers` → `orders` (por `customer_id`) | Quantos pedidos tem cada `customer_id`? |
| b | pessoa (`customer_unique_id`) → `orders` | Quantos pedidos tem cada pessoa? |
| c | `orders` → `items` | Quantos itens tem cada pedido? (inclua pedidos sem itens) |
| d | `orders` → `payments` | Quantos pagamentos tem cada pedido? |
| e | `orders` ↔ `reviews` | Quantas avaliações tem cada pedido? E em quantos pedidos aparece cada `review_id`? |
| f | `products` → `items` e `sellers` → `items` | Algum item aponta para produto ou vendedor inexistente? |

In [7]:
# Dica: use o padrão do aquecimento A.1 para cada relacionamento.
# Para incluir pedidos SEM itens, parta de orders e use LEFT JOIN (junção que mantém
# todas as linhas da tabela da esquerda, mesmo sem correspondência) e COUNT(i.order_id):
# SELECT o.order_id, COUNT(i.order_id) AS n FROM orders o LEFT JOIN items i USING (order_id) GROUP BY 1
# Para (f), use NOT EXISTS, como na Aula 1.

In [18]:
SELECT o.order_id, COUNT(i.order_item_id) AS n_items
FROM orders o
LEFT JOIN items i USING (order_id)
GROUP BY o.order_id;

SyntaxError: invalid syntax (2465473977.py, line 1)

In [20]:
%%sql
SELECT o.order_id, COUNT(i.order_item_id) AS n_items
FROM orders o
LEFT JOIN items i USING (order_id)
GROUP BY o.order_id;

UsageError: Cell magic `%%sql` not found.


In [21]:
sql("""
WITH por_pedido AS (
    SELECT o.order_id, COUNT(i.order_item_id) AS n_items
    FROM orders o
    LEFT JOIN items i USING (order_id)
    GROUP BY o.order_id
)
SELECT MIN(n_items) AS minimo, MAX(n_items) AS maximo, ROUND(AVG(n_items), 2) AS media
FROM por_pedido
""")

,minimo,maximo,media
0,0,21,1.13


### 1b · O grão de `items`
Uma linha de `items` representa uma unidade comprada ou uma linha de pedido com quantidade? Conte quantos pares (pedido, produto) aparecem em mais de uma linha e examine um pedido assim. Repare que não existe coluna de quantidade.

In [6]:
# Dica: GROUP BY order_id, product_id HAVING COUNT(*) > 1
# Para ver um exemplo, pegue um order_id do resultado e filtre items por ele.

In [22]:
sql("""
SELECT order_id, product_id, COUNT(*) AS repeticoes
FROM items
GROUP BY order_id, product_id
HAVING COUNT(*) > 1
""")

,order_id,product_id,repeticoes
0,88d95fe1b916ad9ae117e52dd078e0fb,9f7fa43dfe21e43d37d65a6f2369c6ed,2
1,8921107bab33ea7cd390f1893ac6d643,a8a24b2139731e4be5246104224f0633,2
2,8938715999abb88950de2d1aa4b8f74d,16679d68bdbed3fb97af806424d7f2b7,2
3,893c5d0f1a2fd75917c6b37fcd220601,958cf0f5527f413e08d0cde6bfdbdde2,4
4,893c67d6d4cfdccd40fb76d7c250d7f0,3a7494d56b0e60a130d628caffc99d3f,2
...,...,...,...
7083,886ddb536ba82bd61a9d3ddbf5d0e73d,f5caa36f71f444c6508c84818cab5641,4
7084,887b522c355bcd896b5168f84a11e6f0,bc7783fa09789dd5984d2c9213413a97,2
7085,88834299d278c558dc9d76e61f0c2388,ddb67d084003da7eb28d3e2992a5971c,2
7086,88abc7f56b29139888abe397ab368518,65266b2da20d04dbe00c5c2d3bb7859e,2


In [23]:
sql("""
SELECT *
FROM items
WHERE order_id = 'ID_DO_PEDIDO_AQUI'
""")

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


## 2 · Dependências funcionais e formas normais

### 2a · O endereço do vendedor (3FN)
Em `sellers`, a cidade e o estado estão guardados em cada vendedor. Se o **prefixo de CEP** determinasse o estado (`seller_zip_code_prefix → seller_state`), guardar o estado em cada vendedor seria uma **dependência transitiva** (`seller_id → CEP → estado`), o que viola a 3FN.
1. Teste a dependência `seller_zip_code_prefix → seller_state`. Quantos prefixos aparecem com mais de um estado?
2. Liste esses casos com as cidades e os estados. O que parece ter acontecido? Formule hipóteses, sem concluir.
3. Repita o teste 1 para `customers` (`customer_zip_code_prefix → customer_state` e `→ customer_city`).

In [5]:
# Dica: padrão do aquecimento A.2, trocando a coluna agrupada.
# Para listar: list(seller_state) e list(seller_city) agrupados por prefixo, filtrando os prefixos com violação.

In [24]:
sql("""
SELECT seller_zip_code_prefix, COUNT(DISTINCT seller_state) AS qtd_estados
FROM sellers
GROUP BY seller_zip_code_prefix
HAVING COUNT(DISTINCT seller_state) > 1
""")

,seller_zip_code_prefix,qtd_estados
0,95076,2
1,83020,2
2,37795,2
3,44600,2
4,89052,2
5,88075,2
6,88301,2
7,85960,3
8,80240,2
9,36010,2


In [25]:
sql("""
SELECT seller_zip_code_prefix,
       list(DISTINCT seller_state) AS estados,
       list(DISTINCT seller_city) AS cidades
FROM sellers
GROUP BY seller_zip_code_prefix
HAVING COUNT(DISTINCT seller_state) > 1
""")

,seller_zip_code_prefix,estados,cidades
0,89052,"[SC, SP]",[blumenau]
1,95076,"[SP, RS]",[caxias do sul]
2,37795,"[MG, SP]",[andradas]
3,44600,"[SP, BA]",[ipira]
4,83020,"[SP, PR]",[sao jose dos pinhais]
5,21210,"[RN, RJ]",[rio de janeiro]
6,22783,"[RJ, SP]",[rio de janeiro]
7,37540,"[SP, MG]","[santa rita do sapucai, sao paulo]"
8,81020,"[PR, SP]",[curitiba]
9,81560,"[SP, PR]",[curitiba]


### 2b · O preço do item (2FN)
Em `items`, a chave é composta: (`order_id`, `order_item_id`). Se o preço dependesse **só do produto** (`product_id → price`), ele deveria morar em `products`, e não em `items`.
Teste a dependência `product_id → price`. Quantos produtos foram vendidos por mais de um preço? A modelagem atual está certa?

In [4]:
# Dica: GROUP BY product_id HAVING COUNT(DISTINCT price) > 1

In [26]:
sql("""
SELECT product_id, COUNT(DISTINCT price) AS qtd_precos
FROM items
GROUP BY product_id
HAVING COUNT(DISTINCT price) > 1
""")

,product_id,qtd_precos
0,89321f94e35fc6d7903d36f74e351d40,5
1,eeb2d1e155818957df60a76fc47e75a8,3
2,88a3a882d28c386d79c18ac2141eaea1,4
3,d4eb057c3dd5e4c9513bb461e9bb8ae1,2
4,0bcc3eeca39e1064258aa1e932269894,5
...,...,...
5895,a8366756c95286fc31f1eb653c12ce3a,2
5896,f152b09d0630df240acc7c63009e814d,2
5897,19936fa4f614ee0590d3b77ac83fd648,2
5898,7046a5632fbe7baa273b14b3253e1547,2


### 2c · Provocando uma anomalia de atualização (2FN)
Junte `orders` e `items` numa tabela única e **desnormalizada** (`pedidos_itens`), com uma linha por item e o status do pedido repetido em cada linha.
1. Crie a tabela com `CREATE TABLE pedidos_itens AS SELECT ...`.
2. Simule um sistema que atualiza o status olhando só **uma** linha: mude para `'canceled'` apenas a linha `order_item_id = 1` de todos os pedidos com mais de um item.
3. Conte quantos pedidos agora têm mais de um status. Qual dependência causou o problema e qual forma normal ela viola?

In [3]:
# Dica:
# CREATE OR REPLACE TABLE pedidos_itens AS
#   SELECT i.*, o.order_status FROM items i JOIN orders o USING (order_id)
# UPDATE pedidos_itens SET order_status = 'canceled'
#   WHERE order_item_id = 1 AND order_id IN (SELECT order_id FROM pedidos_itens GROUP BY 1 HAVING COUNT(*) > 1)

In [27]:
sql("""
CREATE OR REPLACE TABLE pedidos_itens AS
SELECT i.*, o.order_status
FROM items i
JOIN orders o USING (order_id)
""")

AttributeError: 'NoneType' object has no attribute 'df'

In [30]:
con.execute("""
CREATE OR REPLACE TABLE pedidos_itens AS
SELECT i.*, o.order_status
FROM items i
JOIN orders o USING (order_id)
""")

In [28]:
sql("""
UPDATE pedidos_itens
SET order_status = 'canceled'
WHERE order_item_id = 1
  AND order_id IN (
      SELECT order_id
      FROM pedidos_itens
      GROUP BY 1
      HAVING COUNT(*) > 1
  )
""")

AttributeError: 'NoneType' object has no attribute 'df'

In [31]:
con.execute("""
UPDATE pedidos_itens
SET order_status = 'canceled'
WHERE order_item_id = 1
  AND order_id IN (
      SELECT order_id
      FROM pedidos_itens
      GROUP BY 1
      HAVING COUNT(*) > 1
  )
""")

In [29]:
sql("""
SELECT order_id, COUNT(DISTINCT order_status) AS qtd_status
FROM pedidos_itens
GROUP BY order_id
HAVING COUNT(DISTINCT order_status) > 1
""")

,order_id,qtd_status
0,001ab0a7578dd66cd4b0a71f5b6e1e41,2
1,001d8f0e34a38c37f7dba2a37d4eba8b,2
2,003822434f91204da0a51fe4cf2aba18,2
3,003f201cdd39cdd59b6447cff2195456,2
4,005d9a5423d47281ac463a968b3936fb,2
...,...,...
9757,ffd6f465cab015b452d398ac1f5d978a,2
9758,ffd84ab39cd5e873d8dba24342e65c01,2
9759,ffe4b41e99d39f0b837a239110260530,2
9760,ffecd5a79a0084f6a592288c67e3c298,2


## 3 · Restrições: o banco como guardião
Os arquivos CSV não impõem nenhuma regra. Vamos criar um esquema com PK e FK e carregar os dados nele.
1. Crie `t_customers` (PK `customer_id`) e `t_orders` (PK `order_id`, FK `customer_id` → `t_customers`). Carregue ambas. Corrija o CEP do cliente com `lpad` (preenche com zeros à esquerda), como visto na Aula 1.
2. Crie `t_reviews` com PK em `review_id` e FK `order_id` → `t_orders`. Tente carregar. O que acontece? Quantas linhas entram?
3. Proponha e implemente uma chave que funcione para `t_reviews`. O que ela revela sobre a cardinalidade entre pedidos e avaliações?
4. Prove que a FK protege: tente inserir em `t_orders` um pedido com `customer_id` inexistente.
5. Por que não é possível criar uma PK em `geolocation` pelo prefixo de CEP? Proponha, em texto, uma tabela que permita essa PK.

In [13]:
# Dica: modelo de DDL
# con.execute("""CREATE OR REPLACE TABLE t_customers (
#     customer_id VARCHAR PRIMARY KEY, customer_unique_id VARCHAR NOT NULL,
#     zip_prefix VARCHAR(5), city VARCHAR, state VARCHAR(2))""")
# con.execute("INSERT INTO t_customers SELECT customer_id, customer_unique_id, "
#             "lpad(CAST(customer_zip_code_prefix AS VARCHAR), 5, '0'), customer_city, customer_state FROM customers")
# Na FK: order_id VARCHAR REFERENCES t_orders(order_id)
# Use tentar(...) para os comandos que podem ser recusados.
# Atenção: no DuckDB, crie as tabelas na ordem pai -> filho e, se precisar recriar,
# apague primeiro os filhos (DROP TABLE IF EXISTS t_reviews).

In [32]:
con.execute("DROP TABLE IF EXISTS t_reviews")
con.execute("DROP TABLE IF EXISTS t_orders")
con.execute("DROP TABLE IF EXISTS t_customers")

con.execute("""
CREATE TABLE t_customers (
    customer_id VARCHAR PRIMARY KEY,
    customer_unique_id VARCHAR NOT NULL,
    zip_prefix VARCHAR(5),
    city VARCHAR,
    state VARCHAR(2)
)
""")

con.execute("""
INSERT INTO t_customers
SELECT customer_id, customer_unique_id,
       lpad(CAST(customer_zip_code_prefix AS VARCHAR), 5, '0'),
       customer_city, customer_state
FROM customers
""")

con.execute("""
CREATE TABLE t_orders (
    order_id VARCHAR PRIMARY KEY,
    customer_id VARCHAR REFERENCES t_customers(customer_id),
    order_status VARCHAR
)
""")

con.execute("""
INSERT INTO t_orders
SELECT order_id, customer_id, order_status
FROM orders
""")

In [33]:
con.execute("""
CREATE TABLE t_reviews (
    review_id VARCHAR PRIMARY KEY,
    order_id VARCHAR REFERENCES t_orders(order_id)
)
""")

tentar("""
INSERT INTO t_reviews
SELECT review_id, order_id
FROM reviews
""")

RECUSADO PELO BANCO: ConstraintException
Constraint Error: PRIMARY KEY or UNIQUE constraint violation: duplicate key "3242cc306a9218d0377831e175d62fbf"


In [34]:
con.execute("DROP TABLE t_reviews")

con.execute("""
CREATE TABLE t_reviews (
    review_id VARCHAR,
    order_id VARCHAR REFERENCES t_orders(order_id),
    PRIMARY KEY (review_id, order_id)
)
""")

con.execute("""
INSERT INTO t_reviews
SELECT review_id, order_id
FROM reviews
""")

In [35]:
tentar("""
INSERT INTO t_orders (order_id, customer_id, order_status)
VALUES ('pedido_falso_123', 'cliente_inexistente_999', 'delivered')
""")

RECUSADO PELO BANCO: ConstraintException
Constraint Error: Violates foreign key constraint because key "customer_id: cliente_inexistente_999" does not exist in the referenced table


## 4 · O pedido como documento (NoSQL)
Num banco de documentos, a pergunta de modelagem muda: em vez de "como evitar redundância?", passa a ser "como a aplicação lê o dado?". Se a tela do pedido mostra cliente, itens e pagamentos juntos, faz sentido **embutir** tudo num só documento.
1. Crie a tabela `pedidos_doc` com uma linha por pedido e uma coluna `doc` em JSON contendo: dados do pedido, um objeto `cliente` e as listas `itens` e `pagamentos`.
2. Mostre o documento de um pedido com 3 itens, formatado.
3. Consultando **só os documentos**, calcule o total de itens e o faturamento (soma dos preços). Confira contra a tabela `items`.
4. Quantos documentos têm a lista de itens vazia ou nula? De onde vêm?
5. Qual o tamanho total dos documentos em MB? Compare com o tamanho dos CSVs de `orders`, `customers`, `items` e `payments` juntos.

In [14]:
# Dica: para cada pedido, uma subconsulta monta a lista de itens:
# (SELECT list(struct_pack(item := i.order_item_id, produto := i.product_id, preco := i.price)
#          ORDER BY i.order_item_id)
#    FROM items i WHERE i.order_id = o.order_id) AS itens
# O documento inteiro: to_json(struct_pack(order_id := o.order_id, ..., cliente := struct_pack(...), itens := (...)))
# Para somar preços dentro do JSON: FROM pedidos_doc, json_each(doc, '$.itens') x  e  x.value ->> 'preco'
# (json_each "desdobra" a lista em linhas; ->> extrai um campo como texto; converta com CAST(... AS DOUBLE))
# Tamanho dos arquivos: os.path.getsize(caminho) em bytes.

In [36]:
con.execute("""
CREATE OR REPLACE TABLE pedidos_doc AS
SELECT o.order_id,
       to_json(struct_pack(
           order_id := o.order_id,
           status := o.order_status,
           cliente := (SELECT struct_pack(
                           customer_id := c.customer_id,
                           unique_id := c.customer_unique_id,
                           zip := c.customer_zip_code_prefix
                       )
                       FROM customers c WHERE c.customer_id = o.customer_id),
           itens := (SELECT list(struct_pack(
                           item := i.order_item_id,
                           produto := i.product_id,
                           preco := i.price
                       ) ORDER BY i.order_item_id)
                     FROM items i WHERE i.order_id = o.order_id),
           pagamentos := (SELECT list(struct_pack(
                               seq := p.payment_sequential,
                               tipo := p.payment_type,
                               valor := p.payment_value
                           ) ORDER BY p.payment_sequential)
                          FROM payments p WHERE p.order_id = o.order_id)
       )) AS doc
FROM orders o
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [37]:
sql("""
SELECT doc
FROM pedidos_doc
WHERE json_array_length(json_extract(doc, '$.itens')) = 3
LIMIT 1
""")

,doc
0,"{""order_id"":""b3111cd50c7a34635433d6a557a6bf64""..."


In [38]:
sql("""
SELECT COUNT(x.value) AS total_itens,
       ROUND(SUM(CAST(x.value ->> 'preco' AS DOUBLE)), 2) AS faturamento_total
FROM pedidos_doc, json_each(doc, '$.itens') x
""")

,total_itens,faturamento_total
0,113425,13591643.7


In [39]:
sql("""
SELECT COUNT(*) AS qtd_nulos
FROM pedidos_doc
WHERE json_extract(doc, '$.itens') IS NULL OR doc ->> '$.itens' = '[]'
""")

ConversionException: Conversion Error: Failed to cast value to numerical: {"order_id":"903b9800ae4c27c2f80b83ea286fb0f8","st...

LINE 1: ... FROM pedidos_doc WHERE ((((json_extract(doc, '$.itens') IS NULL) OR doc) ->> '$.itens') = '[]')
                                                                        ^

In [40]:
sql("""
SELECT COUNT(*) AS qtd_nulos
FROM pedidos_doc
WHERE json_extract(doc, '$.itens') IS NULL OR (doc ->> '$.itens') = '[]'
""")

,qtd_nulos
0,0


In [41]:
import os

# Altera 'caminho_dos_arquivos' para a pasta onde tens os CSVs guardados.
# Exemplo de utilização da dica do os.path.getsize:
# tamanho_orders = os.path.getsize('orders.csv')

### 4b · Embutir tem preço
No documento, os dados do cliente ficam **dentro de cada pedido**.
1. Quantas pessoas (`customer_unique_id`) aparecem com mais de um prefixo de CEP?
2. Se uma pessoa muda de endereço, quantos documentos precisam ser alterados num modelo que embute o cliente? E no modelo relacional?
3. **Hipótese a discutir, não a concluir:** por que a Olist criaria um `customer_id` novo a cada pedido? Que vantagem isso daria ao registrar o endereço de entrega?

In [15]:
# Dica: GROUP BY customer_unique_id HAVING COUNT(DISTINCT customer_zip_code_prefix) > 1

In [42]:
sql("""
SELECT customer_unique_id, COUNT(DISTINCT customer_zip_code_prefix) AS qtd_ceps
FROM customers
GROUP BY customer_unique_id
HAVING COUNT(DISTINCT customer_zip_code_prefix) > 1
""")

,customer_unique_id,qtd_ceps
0,b3a6549b3cd47a7117094faafce8b10d,2
1,2c6a91479a7dc00d8c9d650d8dee88ca,2
2,eb89ecac165940c95a05cb77e89a789d,2
3,50ad9fb3ba7f8cf336529fe94a182996,2
4,e8663abca66edfe7b18d03a921637578,2
...,...,...
245,97981245c3257ea9b14befffd560177b,2
246,4cfa5155cf7cff8eb15e0b12041d058e,2
247,fe3e52de024b82706717c38c8e183084,2
248,5ad84888519e10ff52d8af00641d41bc,2


## 5 · Qual NoSQL para qual necessidade? (discussão, sem código)
Para cada necessidade, indique o tipo de banco mais adequado (**documento**, **chave-valor**, **colunar** ou **grafo**) e justifique em uma frase.

| Necessidade | Tipo | Justificativa |
|---|---|---|
| Consultar o status atual de um pedido pelo `order_id`, milhares de vezes por segundo | | |
| Catálogo de produtos em que cada categoria tem atributos diferentes | | |
| Recomendação "quem comprou este produto também comprou..." | | |
| Gravar bilhões de eventos de rastreamento de entrega, lidos por pedido e por data | | |

In [16]:
# Esta seção é de discussão. Escreva suas respostas na tabela acima (clique duas vezes na célula de texto para editar).

## Modelo do Grupo (entregue ao fim da prática)
Com base nos exercícios, registre o modelo lógico que o grupo propõe para o núcleo da Olist. Uma linha por tabela.

| Tabela | PK | FKs | Grão (uma linha representa...) | Ajuste em relação ao CSV original |
|---|---|---|---|---|
| customers | | | | |
| orders | | | | |
| items | | | | |
| payments | | | | |
| reviews | | | | |
| products | | | | |
| sellers | | | | |
| geolocation | | | | |